# Notebook 2b – Tấn công có mục tiêu (Targeted Attack)

**Đề tài 2.21 – Nghiên cứu và phòng thủ trước tấn công đối kháng lên mô hình phân loại ảnh**

Mục nâng cao của Giai đoạn 3: buộc mô hình phân loại ảnh **mèo thành chó** và so sánh độ khó với tấn công không mục tiêu.

- Tấn công: PGD-10 targeted (`targeted=True`, `y` = nhãn đích, cập nhật theo chiều **giảm** loss), α = 2/255, random start, cùng các ε trong `config.EPSILONS`.
- **Thí nghiệm A – mèo → chó:** các ảnh *cat* trong tập đánh giá mà baseline đoán đúng. Đo tỷ lệ bị đoán thành *dog* (targeted) và tỷ lệ bị đoán sai bất kỳ (untargeted) trên cùng ảnh.
- **Thí nghiệm B – nhãn đích ngẫu nhiên:** mọi ảnh baseline đoán đúng trong 1.000 ảnh đánh giá, mỗi ảnh một nhãn đích ngẫu nhiên khác nhãn thật (seed 42). So với ASR untargeted đã đo ở Notebook 2 (`results/attack_metrics.json`).

Đầu ra: `results/targeted_metrics.json`, `results/targeted_success.png`, `results/targeted_examples_cat_to_dog.png`.
Chạy được trên CPU (~15 phút với 4 luồng), không cần huấn luyện.

## 0. Chuẩn bị môi trường

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Eastern2k4/adversarial-attack-defense.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    Path("/content/drive/MyDrive/AdvML_Project").mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/adversarial-attack-defense")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    cwd = Path.cwd().resolve()
    REPO_DIR = next(
        (root for root in (cwd, *cwd.parents)
         if (root / "src/config.py").is_file() and (root / "notebooks").is_dir()),
        None,
    )
    if REPO_DIR is None:
        raise FileNotFoundError("Hãy mở notebook từ thư mục repo hoặc notebooks/.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
import json
import time
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import torch

from src import config
from src.attacks import pgd
from src.data import get_eval_subset
from src.models import build_model
from src.utils import load_checkpoint, set_seed

device = config.get_device()
if device.type == "cpu":
    torch.set_num_threads(min(torch.get_num_threads(), 4))
print("Device:", device, "| PyTorch:", torch.__version__)

## 1. Nạp baseline và tập đánh giá chung

In [ ]:
PROJECT_DIRS = [config.DRIVE_PROJECT_DIR] if IN_COLAB else [REPO_DIR.parent / "AdvML_Project"]


def find_checkpoint(name, required=True):
    """Tìm checkpoint trong repo/checkpoints trước, sau đó trong AdvML_Project/checkpoints."""
    candidates = [config.CHECKPOINT_DIR / name, *(root / "checkpoints" / name for root in PROJECT_DIRS)]
    path = next((p for p in candidates if p.is_file()), None)
    if path is None and required:
        raise FileNotFoundError(f"Không tìm thấy {name}. Đã kiểm tra:\n" + "\n".join(map(str, candidates)))
    return path


def find_data_dir():
    candidates = [config.DATA_DIR, *(root / "cifar-10-python" for root in PROJECT_DIRS),
                  *(root / "data" for root in PROJECT_DIRS), *PROJECT_DIRS]
    return next((p for p in candidates if (p / "cifar-10-batches-py/test_batch").is_file()), config.DATA_DIR)


DATA_DIR = find_data_dir()
x_eval, y_eval = get_eval_subset(
    as_tensors=True, data_dir=DATA_DIR,
    download=not (DATA_DIR / "cifar-10-batches-py/test_batch").is_file(),
    num_workers=config.NUM_WORKERS if device.type == "cuda" else 0,
)


def load_model(name, required=True):
    path = find_checkpoint(name, required)
    if path is None:
        return None
    m = build_model().to(device)
    load_checkpoint(path, m, map_location=device)
    m.eval().requires_grad_(False)
    print("Đã nạp", path)
    return m


model = load_model(config.BASELINE_CKPT)
CAT, DOG = config.CLASSES.index("cat"), config.CLASSES.index("dog")
EPS_KEYS = [f"{round(e * 255)}/255" for e in config.EPSILONS]


@torch.no_grad()
def predict_probs(m, x, batch_size=256):
    out = [m(x[i:i + batch_size].to(device)).softmax(1).cpu() for i in range(0, len(x), batch_size)]
    return torch.cat(out)


def run_pgd(m, x, y, eps, targeted, batch_size=128):
    """PGD-10 theo batch, trả về ảnh đối kháng trên CPU."""
    out = []
    for i in range(0, len(x), batch_size):
        xb, yb = x[i:i + batch_size].to(device), y[i:i + batch_size].to(device)
        out.append(pgd(m, xb, yb, eps=eps, alpha=config.PGD_ALPHA, steps=config.PGD_STEPS,
                       random_start=True, targeted=targeted).cpu())
    return torch.cat(out)


clean_probs = predict_probs(model, x_eval)
clean_pred = clean_probs.argmax(1)
correct = clean_pred.eq(y_eval)
print(f"Clean accuracy: {100 * correct.float().mean():.1f}% ({int(correct.sum())}/{len(y_eval)})")

## 2. Thí nghiệm A – buộc ảnh mèo thành chó

Chỉ dùng các ảnh *cat* mà baseline đoán đúng, để "thành công" phản ánh đúng tác động của nhiễu.
- **Targeted thành công**: dự đoán sau tấn công = *dog*.
- **Untargeted thành công**: dự đoán sau tấn công ≠ *cat* (sai thành lớp bất kỳ).

In [ ]:
cat_idx = torch.where((y_eval == CAT) & correct)[0]
x_cat, y_cat = x_eval[cat_idx], y_eval[cat_idx]
y_dog = torch.full_like(y_cat, DOG)
print(f"Số ảnh cat baseline đoán đúng: {len(cat_idx)}")

exp_a = {"n_images": len(cat_idx), "targeted_success": {}, "untargeted_success": {},
         "targeted_mean_dog_prob": {}, "untargeted_pred_distribution": {}}
adv_cat_8 = None
for eps, key in zip(config.EPSILONS, EPS_KEYS):
    set_seed(config.SEED)
    x_t = run_pgd(model, x_cat, y_dog, eps, targeted=True)
    p_t = predict_probs(model, x_t)
    set_seed(config.SEED)
    x_u = run_pgd(model, x_cat, y_cat, eps, targeted=False)
    p_u = predict_probs(model, x_u)
    exp_a["targeted_success"][key] = 100 * p_t.argmax(1).eq(DOG).float().mean().item()
    exp_a["untargeted_success"][key] = 100 * p_u.argmax(1).ne(CAT).float().mean().item()
    exp_a["targeted_mean_dog_prob"][key] = 100 * p_t[:, DOG].mean().item()
    counts = torch.bincount(p_u.argmax(1), minlength=config.NUM_CLASSES)
    exp_a["untargeted_pred_distribution"][key] = {c: int(n) for c, n in zip(config.CLASSES, counts)}
    if eps == max(config.EPSILONS):
        adv_cat_8, probs_cat_8 = x_t, p_t
    print(f"eps {key:>6s} | targeted cat→dog {exp_a['targeted_success'][key]:5.1f}% "
          f"| untargeted (≠cat) {exp_a['untargeted_success'][key]:5.1f}%")

Phân bố dự đoán của tấn công **untargeted** theo từng ε: ảnh mèo bị đẩy sang lớp nào nhiều nhất? Ở ε nhỏ, nhiễu chỉ đủ vượt qua ranh giới **gần nhất**, nên lớp chiếm tỷ lệ cao nhất cho biết lớp nào "gần" *cat* nhất trong không gian đặc trưng của mô hình. Ở ε lớn, ảnh bị đẩy đi xa hơn nên phân bố rải đều hơn.

In [ ]:
for key in EPS_KEYS:
    dist = exp_a["untargeted_pred_distribution"][key]
    fooled = exp_a["n_images"] - dist["cat"]
    top = [(c, n) for c, n in sorted(dist.items(), key=lambda kv: -kv[1]) if c != "cat" and n][:3]
    print(f"eps {key:>6s} | bị đánh lừa {fooled:3d}/{exp_a['n_images']} | "
          + ", ".join(f"{c} {n} ({100 * n / max(fooled, 1):.0f}%)" for c, n in top))

## 3. Thí nghiệm B – nhãn đích ngẫu nhiên trên mọi ảnh

Mỗi ảnh baseline đoán đúng được gán một nhãn đích ngẫu nhiên ≠ nhãn thật (cố định theo seed).
Targeted thành công khi dự đoán = nhãn đích. So sánh với ASR untargeted cùng cấu hình PGD-10 trong `results/attack_metrics.json` (Notebook 2).

In [ ]:
g = torch.Generator().manual_seed(config.SEED)
offset = torch.randint(1, config.NUM_CLASSES, (len(y_eval),), generator=g)
y_target_all = (y_eval + offset) % config.NUM_CLASSES          # luôn khác nhãn thật
idx_b = torch.where(correct)[0]
x_b, y_b, t_b = x_eval[idx_b], y_eval[idx_b], y_target_all[idx_b]

exp_b = {"n_images": len(idx_b), "targeted_success": {}, "targeted_any_wrong": {}, "time_sec": {}}
for eps, key in zip(config.EPSILONS, EPS_KEYS):
    set_seed(config.SEED)
    t0 = time.perf_counter()
    x_t = run_pgd(model, x_b, t_b, eps, targeted=True)
    exp_b["time_sec"][key] = time.perf_counter() - t0
    pred = predict_probs(model, x_t).argmax(1)
    exp_b["targeted_success"][key] = 100 * pred.eq(t_b).float().mean().item()
    exp_b["targeted_any_wrong"][key] = 100 * pred.ne(y_b).float().mean().item()
    print(f"eps {key:>6s} | đúng nhãn đích {exp_b['targeted_success'][key]:5.1f}% | "
          f"sai bất kỳ {exp_b['targeted_any_wrong'][key]:5.1f}% | {exp_b['time_sec'][key]:.0f}s")

attack_metrics_path = config.RESULTS_DIR / "attack_metrics.json"
untargeted_asr = None
if attack_metrics_path.exists():
    untargeted_asr = json.loads(attack_metrics_path.read_text(encoding="utf-8"))["asr"]["pgd"]
    for key in EPS_KEYS:
        print(f"  untargeted PGD-10 ASR (Notebook 2) eps {key}: {untargeted_asr[key]:.1f}%")

## 4. Biểu đồ và ảnh minh họa

In [ ]:
BLUE, ORANGE, INK2, GRID = "#2a78d6", "#eb6834", "#52514e", "#e4e3df"
x_ticks = [round(e * 255) for e in config.EPSILONS]


def style(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.grid(color=GRID); ax.set_axisbelow(True)
    ax.set_xticks(x_ticks); ax.set_xticklabels(EPS_KEYS)
    ax.set_ylim(0, 105); ax.set_xlabel("ε (L∞)", color=INK2)


fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
ax = axes[0]
ax.plot(x_ticks, [exp_a["untargeted_success"][k] for k in EPS_KEYS], "-o", color=BLUE, lw=2, label="Untargeted (≠ cat)")
ax.plot(x_ticks, [exp_a["targeted_success"][k] for k in EPS_KEYS], "-o", color=ORANGE, lw=2, label="Targeted (= dog)")
ax.set_title(f"A. Ảnh mèo → chó ({exp_a['n_images']} ảnh)", loc="left")
ax.set_ylabel("Tỷ lệ tấn công thành công (%)", color=INK2)
style(ax); ax.legend(frameon=False)
ax = axes[1]
if untargeted_asr is not None:
    ax.plot(x_ticks, [untargeted_asr[k] for k in EPS_KEYS], "-o", color=BLUE, lw=2, label="Untargeted (Notebook 2)")
ax.plot(x_ticks, [exp_b["targeted_success"][k] for k in EPS_KEYS], "-o", color=ORANGE, lw=2, label="Targeted (nhãn đích ngẫu nhiên)")
ax.set_title(f"B. Mọi lớp, nhãn đích ngẫu nhiên ({exp_b['n_images']} ảnh)", loc="left")
style(ax); ax.legend(frameon=False)
fig.suptitle("PGD-10 trên baseline: targeted khó hơn untargeted ở ε nhỏ", x=0.01, ha="left")
plt.tight_layout()
plt.savefig(config.RESULTS_DIR / "targeted_success.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Ảnh minh họa: 4 ảnh mèo baseline tự tin nhất nhưng bị biến thành "dog" ở ε = 8/255
success = torch.where(probs_cat_8.argmax(1) == DOG)[0]
success = success[clean_probs[cat_idx[success], CAT].argsort(descending=True)][:4]
fig, axes = plt.subplots(len(success), 3, figsize=(7, 2.4 * len(success)))
axes = np.atleast_2d(axes)
for row, i in enumerate(success):
    x0, xa = x_cat[i], adv_cat_8[i]
    delta = xa - x0
    noise = (delta / (2 * delta.abs().max().clamp_min(1e-12)) + 0.5).clamp(0, 1)
    p0 = clean_probs[cat_idx[i]]
    panels = [(x0, f"Gốc: cat {100 * p0[CAT]:.1f}%"),
              (noise, f"Nhiễu (phóng đại)\nL∞ = {delta.abs().max() * 255:.1f}/255"),
              (xa, f"Đối kháng: dog {100 * probs_cat_8[i, DOG]:.1f}%")]
    for ax, (img, title) in zip(axes[row], panels):
        ax.imshow(img.permute(1, 2, 0).numpy(), interpolation="nearest")
        ax.set_title(title, fontsize=9); ax.axis("off")
plt.suptitle("Targeted PGD-10, ε = 8/255: buộc baseline nhận mèo thành chó")
plt.tight_layout()
plt.savefig(config.RESULTS_DIR / "targeted_examples_cat_to_dog.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Lưu kết quả

In [ ]:
metrics = {
    "model": "baseline_resnet18",
    "attack": {"name": "PGD", "targeted": True, "steps": config.PGD_STEPS, "alpha": config.PGD_ALPHA,
               "random_start": True, "norm": "Linf", "epsilons": EPS_KEYS, "seed": config.SEED},
    "clean_acc": 100 * correct.float().mean().item(),
    "cat_to_dog": exp_a,
    "random_target": exp_b,
    "untargeted_pgd_asr_reference": untargeted_asr,
    "device": str(device),
    "torch": torch.__version__,
}
path = config.RESULTS_DIR / "targeted_metrics.json"
path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print("Đã lưu", path)

## 6. Nhận xét

- Ở cùng ε, tấn công targeted có tỷ lệ thành công **thấp hơn** untargeted: đẩy ảnh qua *một* ranh giới quyết định cụ thể khó hơn qua *bất kỳ* ranh giới nào. Khoảng cách lớn nhất ở ε nhỏ (ε = 1/255: khoảng 8% so với 44% với nhãn đích ngẫu nhiên).
- Khi ε tăng, khoảng cách thu hẹp; ở ε = 8/255 kẻ tấn công gần như luôn ép được baseline trả về **đúng nhãn mình chọn**.
- Cặp *cat → dog* dễ hơn nhiều so với nhãn đích ngẫu nhiên (ε = 1/255: khoảng 34% so với 8%), vì *dog* là lớp gần *cat* nhất: khi tấn công untargeted ở ε nhỏ, *dog* là lớp nhận nhiều ảnh mèo bị đánh lừa nhất (mục 2).